# Chapter 2 — Pre-Model Workflow and Data Preprocessing

> Reproduction notebook for **scikit-learn Cookbook, Third Edition** (John Sukup, Packt 2025).

## Chapter Summary

"Garbage in, garbage out": the quality of the data is the most important factor in a successful ML model, and preprocessing is an *ongoing* task because data changes over time.

| Recipe | Tools | Purpose |
|---|---|---|
| Impact of raw data | – | Know the common issues: missing data, outliers, categorical variables, scale, data leakage |
| Missing data | `SimpleImputer`, `KNNImputer`, `IterativeImputer` | Fill gaps with estimated values |
| Scaling | `StandardScaler`, `MinMaxScaler`, `Normalizer` | Put features on comparable scales |
| Encoding | `OneHotEncoder`, `LabelEncoder`, `ColumnTransformer` | Turn categories into numbers |
| Pipelines | `Pipeline`, `train_test_split` | Chain steps, avoid leakage |
| Feature engineering | `PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel` | Create and select features |
| Practical exercise | California Housing | A full pipeline ending in an estimator |

## 1. The impact of raw data on model performance

**Theory.** Models learn patterns from data. If the data is noisy, biased or incomplete, the model generalises poorly. Common issues:

* **Missing data** – caused by collection errors, failures or corruption. Most scikit-learn estimators refuse to run with `NaN`.
* **Outliers** – distort statistics. Use z-scores or robust scalers (`RobustScaler`). They can also be *informative* (rare events).
* **Categorical variables** – most algorithms need numbers (`OneHotEncoder`, ...).
* **Feature scaling** – distance-based and gradient-based models (KNN, SVM, linear models) are sensitive to different scales.
* **Data leakage** – information from outside the training set leaks into training, giving over-optimistic scores. Split the data *before* preprocessing.

Useful tools: `Pipeline`, `ColumnTransformer` (different steps for different columns), and `sklearn.feature_selection`.

## 2. Handling missing data

**Theory.** Three imputers are available:

* `SimpleImputer` – replaces `NaN` with a statistic (`mean`, `median`, `most_frequent`, `constant`). Fast, little bias, good for small amounts of missing data.
* `KNNImputer` – looks at the `n_neighbors` most similar rows and uses their values. Captures relationships between features (numeric data).
* `IterativeImputer` – models each feature with missing values as a regression on the other features, in a round-robin way. Handles multivariate relationships; still *experimental*, so it needs `enable_iterative_imputer`.

How to choose: data type, distribution, *pattern* of missingness (random or not), and the *amount* of missing data. Tree-based models (decision trees, Random Forest) do not require imputation.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(2024)  # For reproducibility
n_samples = 20
n_features = 10

# Generate random data
data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
    for i in range(n_features)
}

# Convert it to a DataFrame
df = pd.DataFrame(data)

# Randomly introduce missing values (approximately 20% of the data)
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

display(df.head(6))

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,NaN,42.009814,34.680397,49.962259,NaN,NaN,89.954588,41.152421,NaN
1,69.910875,9.554215,6.436369,31.287816,37.966499,NaN,82.005649,NaN,75.797616,91.382492
2,NaN,96.090974,59.643269,84.710402,NaN,84.109027,NaN,70.288128,1.778343,4.117690
3,NaN,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,NaN,NaN,80.077973
4,20.501895,NaN,89.248639,67.655865,58.635861,78.225721,60.911562,NaN,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,NaN,19.703051,34.423301,93.399494,72.206680,12.640276


### 2.1 `SimpleImputer`

In [2]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,53.864661,42.009814,34.680397,49.962259,57.822964,51.145011,89.954588,41.152421,48.400044
1,69.910875,9.554215,6.436369,31.287816,37.966499,57.822964,82.005649,50.715003,75.797616,91.382492
2,52.558605,96.090974,59.643269,84.710402,46.055883,84.109027,51.145011,70.288128,1.778343,4.117690
3,52.558605,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.715003,60.616723,80.077973
4,20.501895,53.864661,89.248639,67.655865,58.635861,78.225721,60.911562,50.715003,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.055883,19.703051,34.423301,93.399494,72.206680,12.640276


### 2.2 `KNNImputer`

In [3]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)
knn_imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,48.954043,42.009814,34.680397,49.962259,93.910386,52.549271,89.954588,41.152421,59.833678
1,69.910875,9.554215,6.436369,31.287816,37.966499,86.793468,82.005649,45.416191,75.797616,91.382492
2,67.752344,96.090974,59.643269,84.710402,73.338309,84.109027,59.012876,70.288128,1.778343,4.117690
3,47.880864,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,64.510281,39.726833,80.077973
4,20.501895,31.670912,89.248639,67.655865,58.635861,78.225721,60.911562,25.903612,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,59.716773,19.703051,34.423301,93.399494,72.206680,12.640276


### 2.3 `IterativeImputer`

In [4]:
from sklearn.experimental import enable_iterative_imputer  # Experimental feature requires loading
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(random_state=2024)
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)
iterative_imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,54.365008,42.009814,34.680397,49.962259,58.680582,51.178102,89.954588,41.152421,48.292275
1,69.910875,9.554215,6.436369,31.287816,37.966499,60.070634,82.005649,50.710323,75.797616,91.382492
2,52.539676,96.090974,59.643269,84.710402,46.121290,84.109027,51.155371,70.288128,1.778343,4.117690
3,52.630591,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.680265,50.650877,80.077973
4,20.501895,53.198954,89.248639,67.655865,58.635861,78.225721,60.911562,50.691674,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.156310,19.703051,34.423301,93.399494,72.206680,12.640276


**Takeaway.** The three imputers fill the same gaps with slightly different numbers. Simple = fast/lightweight; KNN = uses neighbours; Iterative = uses all other features (most expensive, usually most accurate).

## 3. Scaling techniques

**Theory.** Features can live on very different scales (age 0–100 vs. income 0–100,000). Scaling prevents one feature from dominating.

* **Standardization (z-score)**: mean 0, std 1 → $z=\frac{x-\mu}{\sigma}$. Good for roughly Gaussian data.
* **Min-max scaling (normalisation to a range)**: $x'=\frac{x-x_{min}}{x_{max}-x_{min}}$ → values in [0, 1].
* **Normalizer (L1/L2)**: scales each **sample (row)** to unit norm, $\lVert x\rVert_2=\sqrt{\sum x_i^2}$. Useful for sparse data or when direction matters more than magnitude.

> The book stresses that *standardization* and *normalization* are different things and must not be used interchangeably. Tree-based models do not need scaling.

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)
scaled_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.271931,0.019495,-0.373057,-0.894258,0.188761,0.031844,0.001905,1.683800,-0.780898,-0.003574
1,0.756048,-1.874939,-1.592196,-1.043864,-0.391744,0.085949,1.426871,-0.000534,0.652484,1.429056
2,-0.000939,1.783513,0.231259,1.311954,0.002887,1.021588,0.000854,0.839730,-2.409929,-1.472256
3,0.003022,-1.214477,1.056818,1.458037,-1.411839,1.531340,1.551267,-0.001824,-0.387917,1.053212
4,-1.397054,-0.029802,1.245866,0.559887,0.608499,0.792594,0.451822,-0.001334,0.210478,1.686279
5,-1.828276,0.969036,-1.125549,-2.186891,0.004582,-1.485267,-0.772566,1.831653,0.503915,-1.188904


In [6]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()
minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(minmax_scaled_data, columns=iterative_imputed_df.columns)
minmax_scaled_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.603506,0.517824,0.380540,0.354639,0.569020,0.555896,0.435006,0.962767,0.402156,0.464988
1,0.721357,0.000000,0.000369,0.313594,0.413077,0.571920,0.833074,0.538604,0.756013,0.918562
2,0.537080,1.000000,0.568987,0.959922,0.519088,0.849027,0.434713,0.750206,0.000000,0.000000
3,0.538045,0.180530,0.826426,1.000000,0.139045,1.000000,0.867825,0.538279,0.499171,0.799569
4,0.197218,0.504349,0.885377,0.753589,0.681776,0.781206,0.560692,0.538402,0.646896,1.000000
5,0.092244,0.777371,0.145886,0.000000,0.519543,0.106575,0.218656,1.000000,0.719336,0.089710


In [7]:
from sklearn.preprocessing import Normalizer

normalizer = Normalizer()
normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)
normalized_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339168,0.313579,0.242313,0.200037,0.288183,0.338471,0.295196,0.518860,0.237368,0.278551
1,0.376706,0.051482,0.034682,0.168591,0.204578,0.323683,0.441878,0.273247,0.408426,0.492404
2,0.264336,0.483450,0.300075,0.426192,0.232044,0.423166,0.257371,0.353631,0.008947,0.020717
3,0.243762,0.116607,0.387811,0.407684,0.078213,0.450213,0.392278,0.234729,0.234592,0.370886
4,0.095909,0.248868,0.417511,0.316499,0.274302,0.365945,0.284948,0.237139,0.304609,0.463686
5,0.068115,0.493382,0.128781,0.034471,0.296421,0.126535,0.221071,0.599823,0.463720,0.081177


**Effect on algorithms (from the book):** KNN is sensitive to distance → scaling matters; SVM works best with standardized data; linear models converge faster with standardized features.

## 4. Encoding categorical variables

**Theory.**

* **Nominal** variables have no order (colour, brand) → use **one-hot encoding** (one binary column per category). Beware: many categories produce a huge, sparse matrix.
* **Ordinal** variables have an order (rating 1–5) → integer codes are meaningful.
* `LabelEncoder` gives each category an integer. For *nominal* features this creates an artificial order, which can mislead models. (In practice `LabelEncoder` is meant for the **target**; `OrdinalEncoder` is the feature-level equivalent. The book uses it on features only to demonstrate the effect.)
* `ColumnTransformer` applies different preprocessing to different columns (e.g. scale numeric + one-hot categorical).

In [8]:
np.random.seed(2024)  # for reproducibility
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame(
    {
        "Department": np.random.choice(categories, size=20),
        "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
        "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
    }
)
display(categorical_data.head(6))

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA


### 4.1 `OneHotEncoder`

In [9]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data, columns=onehot_encoder.get_feature_names_out()
)
onehot_encoded_df.head(6)

,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


### 4.2 `LabelEncoder`

In [10]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoded_df = pd.DataFrame()
for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(categorical_data[column])
label_encoded_df.head(6)

,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1


### 4.3 `ColumnTransformer` (mixed numeric + categorical data)

In [11]:
from sklearn.compose import ColumnTransformer

np.random.seed(2024)  # for reproducibility
mixed_data = pd.DataFrame(
    {
        "Age": np.random.randint(25, 65, size=20),
        "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
        "Experience": np.random.randint(1, 20, size=20),
        "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
        "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    }
)
display(mixed_data.head(6))

,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior


In [12]:
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)
transformed_data = column_transformer.fit_transform(mixed_data)

# Get feature names for the transformed columns
numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = (
    column_transformer
    .named_transformers_["cat"]
    .get_feature_names_out(["Department", "Position"])
)

# Create the transformed DataFrame
transformed_df = pd.DataFrame(
    transformed_data, columns=numeric_cols + list(categorical_cols)
)
transformed_df.head(6)

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.045349,-0.305043,0.327303,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.727681,1.105091,1.262454,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636358,-1.581768,1.075424,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358300,-1.577167,-0.607848,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.023186,-0.565241,-0.420818,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.562482,0.852382,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0


## 5. Introduction to pipelines in scikit-learn

**Theory.** A `Pipeline` is a list of `(name, transformer)` steps, optionally ending with an estimator. Advantages: sequential execution, simpler code, consistency between training and prediction, easy hyperparameter tuning (`step__param`), and modularity.

**Why split *before* transforming?**

* It avoids **data leakage**: statistics (mean, std) are computed only from training data.
* It gives a realistic evaluation on unseen data.
* The test set is then transformed with the *training* statistics (`transform`, not `fit_transform`).

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Separate the features and target (the book assumes the last column is the target)
X = transformed_df.iloc[:, :-1]  # all columns except last
y = transformed_df.iloc[:, -1]   # last column

# Split the data BEFORE any fitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

# Create a pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),  # handle missing values
    ("scaler", StandardScaler()),                 # scale the features
])

X_train_transformed = pipeline.fit_transform(X_train)  # fit on train only
X_test_transformed = pipeline.transform(X_test)        # reuse train statistics

# Put the results back into DataFrames to preserve column names
X_train_transformed = pd.DataFrame(X_train_transformed, columns=X_train.columns, index=X_train.index)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=X_test.columns, index=X_test.index)
X_train_transformed.head()

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
13,0.834298,-0.867451,-1.559779,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,-1.0
12,-0.689202,1.282818,-0.150946,-0.57735,2.081666,-0.480384,-0.774597,-0.57735,1.0
16,0.544107,-1.173357,-1.761041,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,1.0
5,-1.342131,0.821525,-0.150946,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,-1.0
17,1.414679,0.611115,0.654101,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,1.0


### Visualising the pipeline

In [14]:
from sklearn import set_config

set_config(display="diagram")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['Age_scaled','Salary_scaled','Experience_scaled',...,'Department_Sales', 'Position_Junior','Position_Manager']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" f

## 6. Feature engineering

**Theory.** Feature engineering = **feature extraction** (create new features from existing ones) + **feature selection** (keep the informative ones). Methods are grouped as *filter* (statistical properties), *wrapper* (train models on subsets), and *hybrid*.

* `PolynomialFeatures(degree=2)` creates powers and interaction terms ($x_1^2$, $x_1x_2$, ...).
* `KBinsDiscretizer` turns continuous variables into bins. Strategies: `uniform` (equal-width bins), `quantile` (equal counts), `kmeans`.
* `RFE` (Recursive Feature Elimination) repeatedly trains a model and removes the least important feature (`RFECV` adds cross-validation).
* `SelectFromModel` keeps features whose importance (e.g. `coef_`) is above a threshold.

In [15]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
    poly_features, columns=poly.get_feature_names_out(X_train_transformed.columns)
)
print(poly_features_df.shape)
poly_features_df.head()

(16, 55)


,1,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,...,Department_IT^2,Department_IT Department_Sales,Department_IT Position_Junior,Department_IT Position_Manager,Department_Sales^2,Department_Sales Position_Junior,Department_Sales Position_Manager,Position_Junior^2,Position_Junior Position_Manager,Position_Manager^2
0,1.0,0.834298,-0.867451,-1.559779,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,-1.0,...,4.333333,-1.612452,-1.20185,-2.081666,0.600000,0.447214,0.774597,0.333333,0.57735,1.0
1,1.0,-0.689202,1.282818,-0.150946,-0.57735,2.081666,-0.480384,-0.774597,-0.57735,1.0,...,0.230769,0.372104,0.27735,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.57735,1.0
2,1.0,0.544107,-1.173357,-1.761041,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,1.0,...,4.333333,-1.612452,-1.20185,2.081666,0.600000,0.447214,-0.774597,0.333333,-0.57735,1.0
3,1.0,-1.342131,0.821525,-0.150946,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,-1.0,...,0.230769,-0.620174,0.27735,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.57735,1.0
4,1.0,1.414679,0.611115,0.654101,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,1.0,...,0.230769,-0.620174,0.27735,-0.480384,1.666667,-0.745356,1.290994,0.333333,-0.57735,1.0


In [16]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")
binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(binned_data, columns=X_train_transformed.columns)
binned_df.head()

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
2,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0
3,0.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,2.0,0.0,2.0


In [17]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(estimator=LinearRegression(), n_features_to_select=1)
rfe.fit(X_train_transformed, y_train)
print(rfe.ranking_)   # 1 = most important

[3 9 8 6 5 4 7 2 1]


In [18]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(
    estimator=LinearRegression(),
    prefit=False,
    threshold="mean",   # use mean of feature importances as threshold
)
selector.fit(X_train_transformed, y_train)
selected_features_mask = selector.get_support()
selected_features = X_train_transformed.columns[selected_features_mask].tolist()
print("Selected:", selected_features)

feature_importance = pd.DataFrame({
    "Feature": X_train_transformed.columns,
    "Importance": selector.estimator_.coef_,
    "Selected": selected_features_mask,
})
feature_importance.sort_values("Importance", key=abs, ascending=False)

Selected: ['Position_Junior', 'Position_Manager']


,Feature,Importance,Selected
8,Position_Manager,-5.000000e-01,True
7,Position_Junior,-4.330127e-01,True
6,Department_Sales,-9.116147e-16,False
3,Department_Finance,-8.700954e-16,False
5,Department_IT,-7.202125e-16,False
4,Department_HR,-5.139679e-16,False
2,Experience_scaled,-1.995702e-16,False
0,Age_scaled,-1.221550e-16,False
1,Salary_scaled,0.000000e+00,False


## 7. Practical exercise — a comprehensive pipeline on California Housing

The book asks the reader to complete an incomplete code block ("Comprehensive Pipeline") with these steps: load California Housing → split → pipeline with **at least three steps** (last = estimator) → fit → evaluate on the test data. The solution below is my own version (the book notes that any transformers/estimators may be used).

Steps used: `SimpleImputer` → `StandardScaler` → `SelectFromModel` (feature selection) → `Ridge` regression.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load the dataset (20,640 records; target = median house value in $100,000)
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)

# 3. Comprehensive pipeline: 4 steps, final step is an estimator
full_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("selector", SelectFromModel(Ridge(alpha=1.0), threshold="median")),
    ("model", Ridge(alpha=1.0)),
])

# 4. Fit the pipeline
full_pipeline.fit(X_train, y_train)

# 5. Evaluate on the test data
y_pred = full_pipeline.predict(X_test)
print(f"Test MSE : {mean_squared_error(y_test, y_pred):.4f}")
print(f"Test R^2 : {r2_score(y_test, y_pred):.4f}")
full_pipeline

---
### Chapter 2 key takeaways
1. Clean and prepare data first: impute, scale, encode.
2. Always **split first**, then fit transformers on the training set only (prevents leakage).
3. `Pipeline` + `ColumnTransformer` bundle everything into one reproducible object.
4. Feature engineering (creating + selecting features) can simplify the model and improve generalisation.